# Logistic Regression Experiment Notebook


Worflow has been split into stages so you can run and inspect each step:
1. Imports and configuration  
2. Data loading and preprocessing  
3. Train–test split  
4. Load best hyperparameters  
5. Build pipeline and train model  
6. Evaluate model and save results  


In [41]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_curve,
    auc,
    confusion_matrix,
)
import numpy as np

In [42]:
# Configuration / paths
DATA_PATH = "../../CSVs/dataset.csv"
PARAM_PATH = "../../MachineLearning/LogReg/best_params.csv"
OUTPUT_DIR = "../../Results/LogRegResults"
RANDOM_STATE = 100
TEST_SIZE = 0.80

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

last_result_csv = os.path.join(OUTPUT_DIR, "../../results_logreg.csv")
best_result_csv = os.path.join(OUTPUT_DIR, "../../results_logreg_best.csv")
roc_data_csv = os.path.join(OUTPUT_DIR, "../../roc_logreg_clean.csv")
summary_csv = os.path.join(OUTPUT_DIR, "../../logreg_summary.csv")

In [43]:
# 1) Load dataset
df = pd.read_csv(DATA_PATH)
y = df["anomaly"]
X = df.drop(columns=["anomaly", "timestamp", "channel", "label"], errors="ignore")

print("Shape of X:", X.shape)
print("Positive class ratio:", y.mean())

Shape of X: (2123, 21)
Positive class ratio: 0.20442769665567592


In [44]:
# 2) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)

Train size: (424, 21) Test size: (1699, 21)


In [45]:
# 3) Load best parameters
p = pd.read_csv(PARAM_PATH).iloc[0].to_dict()
print("Using parameters:", p)

Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}


In [46]:
# 4) Build pipeline
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        solver=p["clf__solver"],
        penalty=p["clf__penalty"],
        C=float(p["clf__C"]),
        class_weight=None if p["clf__class_weight"] == "None" else p["clf__class_weight"],
        max_iter=20000,
        random_state=RANDOM_STATE
    ))
])

pipe

,steps,"[('scaler', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,10.0


In [47]:
# 5) Fit & predict
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

# Classification report as DataFrame
report_dict = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose().round(3)

report_df

,precision,recall,f1-score,support
0,0.953,0.934,0.944,1352.000
1,0.762,0.821,0.791,347.000
accuracy,0.911,0.911,0.911,0.911
macro avg,0.858,0.878,0.867,1699.000
weighted avg,0.914,0.911,0.912,1699.000


In [48]:
# 6) Save last run and update best run

report_df.to_csv(last_result_csv, index=True)
print(f"Saved last run results to {last_result_csv}")

def update_best_result(last_df, best_path):
    if os.path.exists(best_path):
        best_df = pd.read_csv(best_path)
        if "f1-score" in best_df.columns:
            last_mean = last_df["f1-score"].mean()
            best_mean = best_df["f1-score"].mean()
            if last_mean > best_mean:
                print(f"New best model found! (F1 {last_mean:.3f} > {best_mean:.3f})")
                last_df.to_csv(best_path, index=True)
            else:
                print(f"Best model retained (F1 {best_mean:.3f} >= {last_mean:.3f})")
        else:
            last_df.to_csv(best_path, index=True)
    else:
        last_df.to_csv(best_path, index=True)

update_best_result(report_df, best_result_csv)

Saved last run results to ../../Results/LogRegResults\../../results_logreg.csv


In [49]:
# 7) Compute ROC and AUC
if hasattr(pipe.named_steps["clf"], "predict_proba"):
    y_proba = pipe.predict_proba(X_test)[:, 1]
else:
    y_proba = pipe.decision_function(X_test)

fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

roc_df = pd.DataFrame({"fpr": fpr, "tpr": tpr})
roc_df.to_csv(roc_data_csv, index=False)
print(f"Saved ROC data to {roc_data_csv} (AUC = {roc_auc:.3f})")

roc_auc

Saved ROC data to ../../Results/LogRegResults\../../roc_logreg_clean.csv (AUC = 0.933)


0.933167641491738

In [50]:
# 8) Confusion matrix and combined summary
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
cm_df = pd.DataFrame(cm, columns=["Pred 0", "Pred 1"], index=["True 0", "True 1"])

summary_data = {
    "dataset": ["clean"],
    "auc": [roc_auc],
    "accuracy": [report_dict["accuracy"]],
    "precision_0": [report_dict["0"]["precision"]],
    "recall_0": [report_dict["0"]["recall"]],
    "f1_0": [report_dict["0"]["f1-score"]],
    "precision_1": [report_dict["1"]["precision"]],
    "recall_1": [report_dict["1"]["recall"]],
    "f1_1": [report_dict["1"]["f1-score"]],
    "tp": [tp],
    "fp": [fp],
    "tn": [tn],
    "fn": [fn],
}
summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(summary_csv, index=False)
print(f"Saved summary (AUC + Confusion Matrix) to {summary_csv}")

cm_df

Saved summary (AUC + Confusion Matrix) to ../../Results/LogRegResults\../../logreg_summary.csv


,Pred 0,Pred 1
True 0,1263,89
True 1,62,285


In [51]:
# 9) Text overview

print("=== Test Set Classification Report ===")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", cm_df)
print(f"\nAUC: {roc_auc:.3f}")
print("\n=== All results and summaries saved successfully ===")

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.95      0.93      0.94      1352
           1       0.76      0.82      0.79       347

    accuracy                           0.91      1699
   macro avg       0.86      0.88      0.87      1699
weighted avg       0.91      0.91      0.91      1699


Confusion Matrix:
         Pred 0  Pred 1
True 0    1263      89
True 1      62     285

AUC: 0.933

=== All results and summaries saved successfully ===
